In [2]:
import os
import numpy as np
import pandas as pd

# ========= CONFIG =========
BASE = r"C:\Users\loci_\Desktop\trading_webapp\DATA"
IN   = os.path.join(BASE, "all_input_files")

FILES = {
    "RX1": os.path.join(IN, "RX1_small.csv"),
    "AD1": os.path.join(IN, "AD1_small.csv"),
}

# instrument weights
W_INSTR = {"RX1": 0.5, "AD1": 0.5}

# strategy weights inside each instrument
W_STRAT = {"ewma": 0.5, "carry": 0.5}

TRADING_DAYS = 256

# ==== Carry params (per our agreed spec) ====
LOOKBACK = 36
DECAY    = 2.0 / (LOOKBACK + 1.0)
SCALER_CARRY = 30.0
CAP = 20.0
DISTANCE = {"RX1": 3/12, "AD1": 1/12}  # quarterly Bund, monthly AUD

# ==== EWMA params (slow = 4×fast) ====
EWMA_FAST = 4
EWMA_SLOW = 16
EWMA_SCALER = 7.5   # your scale for 4/16
# (If you want a different fast/slow, just swap these and the scaler accordingly.)

# ========= HELPERS =========
def ann_sharpe(x, td=TRADING_DAYS):
    x = pd.Series(x).dropna()
    if len(x) < 3:
        return np.nan
    mu, sd = x.mean(), x.std()
    return np.nan if (sd == 0 or np.isnan(sd)) else (mu / sd) * np.sqrt(td)

def build_carry_raw(df: pd.DataFrame, distance: float) -> pd.Series:
    """Return series of raw-layer PnL = forecast[t] * price_diff[t+1] for carry."""
    d = df.copy()
    d["near"] = d["near"].astype(float)
    d["far"]  = d["far"].astype(float)
    d["return"] = d["near"].diff()
    d["square_returns"] = d["return"] ** 2
    # EWMA variance on squared returns
    var = np.full(len(d), np.nan)
    sq = d["square_returns"].to_numpy()
    i0 = np.argmax(~np.isnan(sq)) if np.any(~np.isnan(sq)) else None
    if i0 is not None:
        var[i0] = sq[i0]
        prev = var[i0]
        for i in range(i0 + 1, len(d)):
            x2 = 0.0 if np.isnan(sq[i]) else sq[i]
            prev = DECAY * x2 + (1 - DECAY) * prev
            var[i] = prev
    d["variance"] = var
    d["annual_std"] = np.sqrt(d["variance"]) * np.sqrt(TRADING_DAYS)

    d["price_difference"]   = d["far"] - d["near"]
    d["net_expected_return"] = d["price_difference"] / float(distance)

    denom = d["annual_std"].replace(0.0, np.nan)
    d["raw_carry"] = d["net_expected_return"] / denom
    d["forecast"]  = d["raw_carry"] * SCALER_CARRY
    d["capped"]    = d["forecast"].clip(-CAP, CAP)
    # raw-layer PnL proxy: today’s forecast * tomorrow’s price_diff
    return (d["capped"] * d["price_difference"].shift(-1)).rename("carry_raw_pnl")

def build_ewma_raw(df: pd.DataFrame, fast: int, slow: int, scaler: float) -> pd.Series:
    """Return series of raw-layer PnL = forecast[t] * ret_pct[t+1] for EWMA crossover."""
    d = df.copy()
    px = d["PX_CLOSE_1D"].astype(float)
    ret_pct = px.pct_change()
    ew_fast = px.ewm(span=fast, adjust=False).mean()
    ew_slow = px.ewm(span=slow, adjust=False).mean()
    raw_cross = ew_fast - ew_slow

    # variance for vol-adjust (as we aligned before): EWMA on squared net returns
    ret_net = px.diff()
    var = np.full(len(d), np.nan)
    arr = ret_net.to_numpy()
    i0 = np.argmax(~np.isnan(arr)) if np.any(~np.isnan(arr)) else None
    if i0 is not None:
        first_sq = (arr[i0] ** 2)
        var[i0] = first_sq
        prev = first_sq
        for i in range(i0 + 1, len(d)):
            x = 0.0 if np.isnan(arr[i]) else arr[i]
            prev = DECAY * (x * x) + (1 - DECAY) * prev
            var[i] = prev

    # Convert to Series for safe division
    vol = pd.Series(np.sqrt(var), index=d.index)
    vol = vol.replace(0.0, np.nan)

    vol_adj = raw_cross / vol
    scaled = (vol_adj * scaler).clip(-CAP, CAP)

    # raw-layer PnL proxy: today’s forecast * tomorrow’s ret (pct)
    return (scaled.shift(1) * ret_pct).rename("ewma_raw_pnl")


# ========= MAIN =========
per_instr_blended = {}
for inst, path in FILES.items():
    if not os.path.exists(path):
        print(f"❌ Missing file for {inst}: {path}")
        continue

    df = pd.read_csv(path)
    # robust date order
    if "Date" in df.columns:
        df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
        df = df.dropna(subset=["Date"]).sort_values("Date").reset_index(drop=True)

    # Column casing fixes:
    # ensure we have PX_CLOSE_1D
    if "PX_CLOSE_1D" not in df.columns and "px_close_1d" in df.columns:
        df["PX_CLOSE_1D"] = df["px_close_1d"]

    # Build both raw series
    carry_raw = build_carry_raw(df, distance=DISTANCE[inst])
    ewma_raw  = build_ewma_raw(df, fast=EWMA_FAST, slow=EWMA_SLOW, scaler=EWMA_SCALER)

    # Align on Date index
    if "Date" in df.columns:
        idx = df["Date"]
        carry_raw.index = idx
        ewma_raw.index  = idx

    # Blend per instrument
    blended = W_STRAT["carry"] * carry_raw.fillna(0.0) + W_STRAT["ewma"] * ewma_raw.fillna(0.0)
    per_instr_blended[inst] = blended.rename(f"{inst}_blend_raw")

    # Print individual strategy Sharpes and blend Sharpe
    print(f"\n[{inst}]")
    print(f"  Carry raw Sharpe: {ann_sharpe(carry_raw):.3f}")
    print(f"  EWMA  raw Sharpe: {ann_sharpe(ewma_raw):.3f}")
    print(f"  Blend raw Sharpe: {ann_sharpe(blended):.3f}")

# Portfolio = weighted sum of instrument blends
# Align on common dates (inner join)
if len(per_instr_blended) == 2:
    dfp = pd.concat(per_instr_blended.values(), axis=1).dropna()
    port = W_INSTR["RX1"] * dfp["RX1_blend_raw"] + W_INSTR["AD1"] * dfp["AD1_blend_raw"]
    print("\n[PORTFOLIO 50% RX1 / 50% AD1 | 50% EWMA / 50% Carry per instrument]")
    print(f"  Portfolio raw Sharpe: {ann_sharpe(port):.3f}")
else:
    print("\n⚠️ Could not build portfolio: missing instrument series.")



[RX1]
  Carry raw Sharpe: 22.401
  EWMA  raw Sharpe: -0.567
  Blend raw Sharpe: 22.288

[AD1]
  Carry raw Sharpe: 22.903
  EWMA  raw Sharpe: -0.917
  Blend raw Sharpe: 22.748

[PORTFOLIO 50% RX1 / 50% AD1 | 50% EWMA / 50% Carry per instrument]
  Portfolio raw Sharpe: 23.166
